# 02 - Evaluation Lab: how well does each rewrite hide AI text from each detector, and what does it cost in meaning?

Each stage saves its results to files, so if Colab disconnects you rerun the cell and it **continues where it stopped**.

1. **Generate**: a small open model writes research-style text three ways: *human* reference (real arXiv abstracts), *unwatermarked* machine text, and *watermarked* text. Watermark types: `kgw` (green-list), `unigram`, `exp` (exponential-minimum sampling) and `synthid` (SynthID-Text, open reference implementation).
2. **Attack**: every watermarked text is rewritten by SIRA (mask the highest-surprisal words), two control strategies (random words, most predictable words) and a plain LLM paraphrase.
3. **Evaluate**: a panel of detectors scores everything. *Watermark detectors* ask "is the watermark still there?" (negatives = unwatermarked text). *Zero-shot detectors* (Fast-DetectGPT, Binoculars) ask "is this machine-written at all?" (negatives = human text).

Do the **smoke** run first (tiny models, a few minutes, numbers meaningless). It exercises every watermark and detector once on a real GPU. Then the **quick** run.

## 0. Setup (same as before)
Set **Runtime -> Change runtime type -> T4 GPU**, and switch on notebook access for `HF_TOKEN`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])

In [ ]:
import os
REPO = "https://github.com/deymonp/sira-lab.git"
if not os.path.exists("/content/sira-lab"):
    !git clone -q {REPO} /content/sira-lab
else:
    !git -C /content/sira-lab pull -q
%cd /content/sira-lab
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## 1. Choose the run
`lab_smoke.yaml` = plumbing check with tiny models. After it works, change the two lines to `configs/lab_quick.yaml` and `/content/runs/quick`.
The folder name is where results are saved. Reusing a folder resumes it, a new folder starts fresh.

In [ ]:
CONFIG = "configs/lab_smoke.yaml"     # later: "configs/lab_quick.yaml"
RUN = "/content/runs/smoke"           # later: "/content/runs/quick"

from sira_lab.config import load_config
from sira_lab.lab.run import run_generate, run_attack, run_evaluate
cfg = load_config(CONFIG)
print("algorithms:", cfg.lab.algorithms, "| attacks:", cfg.lab.attacks, "| texts per class:", cfg.lab.n_samples)

## 2. Stage 1: generate the texts
Downloads the generator model and the arXiv abstracts. If the dataset cannot be loaded, it falls back to a small built-in prompt list and says so.

In [ ]:
run_generate(cfg, RUN)

## 2b. Look at what was generated
`z` is the detector score. Rule of thumb: **z above 4 means "watermark detected"**. Human and unwatermarked text should score near 0, watermarked text clearly higher.

In [ ]:
from sira_lab.lab.io import read_jsonl
from sira_lab.lab.run import load_encoder
from sira_lab.lab.watermark import detect_text
from sira_lab.lab.generate import spec_for

algo = [a for a in cfg.lab.algorithms if a != "synthid"][0]       # synthid is scored by its own detector in stage 3
enc = load_encoder(cfg)
spec = spec_for(cfg.lab, algo)
gen = read_jsonl(f"{RUN}/generated.jsonl")
print(f"z-scores for the '{algo}' watermark. Rule of thumb: z above 4 means watermark detected.\n")
for kind in ("human", "unwatermarked", "watermarked"):
    rows = [r for r in gen if r["kind"] == kind and r["text"].strip() and r["algo"] in ("none", algo)]
    if not rows:
        print(kind.upper(), ": none (no human reference with the builtin prompts)\n"); continue
    r = rows[0]
    print(f"{kind.upper()}  z = {detect_text(r['text'], enc, spec)['z']:.2f}")
    print(r["text"][:400], "\n")

## 3. Stage 2: attack the watermarked texts
Loads the rewriter models (same as notebook 01) and rewrites every watermarked text once per attack. The quick run takes roughly 10 to 25 minutes.

In [ ]:
run_attack(cfg, RUN)

## 4. Stage 3: evaluate
Loads each detector in turn, scores everything, frees it, and moves on. The zero-shot detectors download small language models. Expect a few minutes.

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
df = run_evaluate(cfg, RUN)
display(df.round(3))
display(Image(f"{RUN}/tradeoff.png"))

## How to read the table
| Column | Meaning |
| --- | --- |
| `detector` | `wm_kgw`, `wm_exp`, `wm_synthid`, `wm_unigram` = watermark detectors. `fast_detectgpt`, `binoculars` = zero-shot "is it machine-written?" detectors |
| `attack` | `none` = text with no attack (the starting point). `unwatermarked (AI baseline)` = plain machine text, zero-shot detectors only |
| `neg_set` / `n_neg` | What the detector's false-alarm rate is calibrated on, and how many texts that is. Watermark detectors use unwatermarked machine text; zero-shot detectors use human text. Treat `TPR@FPR` as rough below about 100 |
| `TPR@FPR` | Share of texts still caught when only 1% of negatives are falsely flagged. **Lower after an attack = better evasion** |
| `AUROC` | How well the detector separates these texts from the negatives (1.0 perfect, 0.5 coin flip) |
| `mean_score` | Average detector score (watermark z-score, or zero-shot score where higher = more machine-like) |
| `TPR@thr`, `FPR@thr` | Watermark detectors only: detection and false alarms with the fixed rule z >= 4 |
| `sim_median` | Median meaning similarity to the original (1.0 identical). **The cost side** |
| `edit_mean`, `ppl_ratio_mean` | Share of words kept; how much less natural the text became (1.0 = unchanged) |
| `frac_unchanged`, `frac_best_effort` | Rewrites that failed (text left as is) / that missed the quality gates |

**Things to look for.**
1. SIRA is only interesting if it beats the *random* and *lowest* controls at similar `sim_median`.
2. Watch the zero-shot rows. SIRA swaps surprising words for words a language model finds likely, which can make text look *more* machine-written to a perplexity-based detector. If the zero-shot `TPR@FPR` goes up after SIRA, that is the trade-off we will have to work on.
3. Zero-shot results depend heavily on sampling settings (`temperature`, `top_p` in the config) and on which models the detector uses.

## 5. Save the results
The Colab disk is wiped when the session ends. Run one of these before closing.

In [ ]:
# Option A: download a zip to your computer
!cd /content/runs && zip -qr /content/run_results.zip {RUN.split("/")[-1]}
from google.colab import files
files.download("/content/run_results.zip")

In [ ]:
# Option B: save to a PRIVATE dataset repo on your Hugging Face account (free)
from huggingface_hub import HfApi
api = HfApi()
repo = f"{whoami()['name']}/sira-lab-results"
api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=RUN, repo_id=repo, repo_type="dataset", path_in_repo=RUN.split("/")[-1])
print("saved to https://huggingface.co/datasets/" + repo)